# HST-Integrated COVID-RARS Reliability Run

This notebook controls the tested, restart-safe pipeline. Run cells in order. The three acceptance checkpoints are deliberately manual: candidate generation never authorizes its own result. Validation may select checkpoints and declared fusion rules; test and external labels are evaluation-only. Never rerun or change settings after inspecting held-out outcomes.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

from covid_audio_btp.hst_reliability import (
    find_resumable_detached_run,
    launch_detached_run,
    prepare_hst_prerequisites,
    read_hst_run_progress,
    read_run_status,
    run_preflight,
    wait_for_detached_run,
)

def _find_project_root() -> Path:
    override = os.environ.get("COVID_RARS_PROJECT_ROOT")
    starts = [Path(override).expanduser().resolve()] if override else []
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "configs/hst_reliability.json").is_file() and (candidate / "src/covid_audio_btp").is_dir():
                return candidate
    raise FileNotFoundError("Set COVID_RARS_PROJECT_ROOT to the covid_audio_btp directory")

PROJECT_ROOT = _find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs/hst_reliability.json"
SCIENTIFIC_CONFIG = json.loads(CONFIG_PATH.read_text(encoding="ascii"))
ACCEPTED_FREEZES_PATH = PROJECT_ROOT / "reports/hst/accepted_freezes.json"
CANDIDATE_ROOT = PROJECT_ROOT / "reports/hst/candidates"
APPROVAL_PATH = PROJECT_ROOT / "configs/hst_compare_is10_approval.approved.json"
COMPARATOR_FREEZES_PATH = PROJECT_ROOT / "configs/hst_comparator_accepted_freezes.approved.json"
ENVIRONMENT_LOCK_PATH = PROJECT_ROOT / "configs/hst_environment_lock.approved.json"
FEATURE_TABLE = PROJECT_ROOT / "data/processed/features_compare_is10_merged.csv"
EXPECTED_REMOTE_URL = "https://github.com/nishantharkut/Covid-RARS.git"
MODE = "pilot"
DEVICE = "cuda"
THROUGH = "base_resource_pilot"
THROUGH_MANIFESTS = "manifests"
THROUGH_COMPARATOR = "aligned_comparator"
THROUGH_FINAL = "evidence_pack"
EXPECTED_RUN_ID = "auto"
RESUME_EXISTING_STAGES = True
DETACHED_POLL_INTERVAL_SECONDS = 60.0
DETACHED_STALE_AFTER_SECONDS = 5.0 * 60.0
DETACHED_MAX_WAIT_SECONDS = 8.0 * 24.0 * 60.0 * 60.0
MAX_CONCURRENT_GPU_JOBS = 1
MAX_PROJECTED_SERIAL_GPU_HOURS = 168.0
END_TO_END_OVERHEAD_MULTIPLIER = 1.5
PERFORMANCE_OBJECTIVES = {"cough": 0.868, "breath": 0.842, "speech": 0.891, "cough_speech": 0.897}
if SCIENTIFIC_CONFIG["runtime"]["max_concurrent_gpu_jobs"] != MAX_CONCURRENT_GPU_JOBS:
    raise RuntimeError("This notebook does not permit concurrent GPU jobs")
if SCIENTIFIC_CONFIG["runtime"]["maximum_projected_serial_gpu_hours"] != MAX_PROJECTED_SERIAL_GPU_HOURS:
    raise RuntimeError("The notebook runtime ceiling differs from the frozen configuration")
if SCIENTIFIC_CONFIG["runtime"]["end_to_end_overhead_multiplier"] != END_TO_END_OVERHEAD_MULTIPLIER:
    raise RuntimeError("The notebook runtime overhead differs from the frozen configuration")
if SCIENTIFIC_CONFIG["experiment"]["test_evaluation_policy"] != "single_nonadaptive_pass_of_prespecified_locked_endpoints_after_validation_freeze":
    raise RuntimeError("Held-out evaluation policy is not frozen")
if not SCIENTIFIC_CONFIG["performance_objectives"]["test_set_is_not_a_stopping_rule"]:
    raise RuntimeError("Held-out results must not control training or reruns")
print("The 0.897 multimodal value is an engineering objective, not a guarantee or a test-set stopping rule.")

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _run_script(*arguments: object) -> None:
    subprocess.run([sys.executable, *map(str, arguments)], cwd=PROJECT_ROOT, check=True)

def _print_hst_progress(current: dict[str, object], *, through: str) -> None:
    progress = read_hst_run_progress(
        project_root=PROJECT_ROOT, run_id=str(current["run_id"]), through=through,
    )
    stages = progress["pipeline_stages"]
    training = progress["confirmatory_training"]
    message = (
        f"status={current.get('status')} stage={current.get('stage')} | "
        f"pipeline={stages['completed']}/{stages['total']} ({stages['percent']:.1f}%) | "
        f"durable training={training['durable_job_equivalents']:.3f}/"
        f"{training['total_jobs']} jobs ({training['percent']:.1f}%)"
    )
    job = progress.get("current_job")
    if job is not None:
        message += (
            f" | job={job['job_id']} stage={job['stage']} status={job['status']} "
            f"fold={job['fold']} seed={job['seed']} modality={job['modality']} "
            f"protocol={job['protocol']}"
        )
        if not job["checkpointed"]:
            message += " | awaiting first optimizer-safe checkpoint; durable job fraction=0"
        else:
            message += (
                f" | durable_epochs={job['durable_epochs']:.3f}/{job['max_epochs']} "
                f"({job['epoch_percent']:.1f}%) next=epoch-{job['resume_epoch']} "
                f"batch-{job['next_consumed_batch_index']}/{job['epoch_batch_count']}"
            )
            if job["checkpoint_resume_safe"]:
                message += (
                    f" | durable checkpoint={job['checkpoint_reason']} "
                    f"generation={job['checkpoint_generation']} "
                    f"path={job['checkpoint_path']} "
                    f"pointer={job['checkpoint_pointer_path']} "
                    f"sha256={job['checkpoint_sha256']}"
                )
    print(message, flush=True)

def launch_and_wait(*, mode: str, through: str, expected_run_id: str = "auto", expected_manual_gate: str | None = None):
    if not RESUME_EXISTING_STAGES:
        raise RuntimeError("This notebook only permits checksummed resume execution")
    preflight = run_preflight(
        config_path=CONFIG_PATH, project_root=PROJECT_ROOT, mode=mode, device=DEVICE,
        accepted_freezes_path=ACCEPTED_FREEZES_PATH,
    )
    if preflight["status"] != "ready":
        raise RuntimeError(preflight)
    existing_launch = find_resumable_detached_run(
        project_root=PROJECT_ROOT, run_id=str(preflight["run_id"]),
        mode=mode, device=DEVICE, through=through,
        expected_run_id=expected_run_id,
        stale_after_seconds=DETACHED_STALE_AFTER_SECONDS,
    )
    launch = existing_launch or launch_detached_run(
        config_path=CONFIG_PATH, project_root=PROJECT_ROOT, mode=mode, device=DEVICE,
        through=through, accepted_freezes_path=ACCEPTED_FREEZES_PATH,
        expected_run_id=expected_run_id,
    )
    status = wait_for_detached_run(
        project_root=PROJECT_ROOT, status_id=launch["launch_id"],
        poll_interval_seconds=DETACHED_POLL_INTERVAL_SECONDS,
        stale_after_seconds=DETACHED_STALE_AFTER_SECONDS,
        timeout_seconds=DETACHED_MAX_WAIT_SECONDS,
        on_poll=lambda current: _print_hst_progress(current, through=through),
    )
    _print_hst_progress(status, through=through)
    if expected_manual_gate is not None:
        if status["status"] != "failed" or expected_manual_gate not in str(status.get("error", "")):
            raise RuntimeError(status)
    elif status["status"] != "success":
        raise RuntimeError(status)
    return launch, status


In [ ]:
PREREQUISITES = prepare_hst_prerequisites(config_path=CONFIG_PATH, project_root=PROJECT_ROOT)
PILOT_PREFLIGHT = run_preflight(
    config_path=CONFIG_PATH, project_root=PROJECT_ROOT, mode=MODE, device=DEVICE,
    accepted_freezes_path=ACCEPTED_FREEZES_PATH,
)
if PILOT_PREFLIGHT["status"] != "ready":
    raise RuntimeError(PILOT_PREFLIGHT)
PREREQUISITES, PILOT_PREFLIGHT


In [ ]:
PILOT_LAUNCH, PILOT_STATUS = launch_and_wait(mode=MODE, through=THROUGH)
PILOT_RUN_ROOT = PROJECT_ROOT / "data/outputs/hst" / PILOT_STATUS["run_id"]
PILOT_STATUS


In [ ]:
CANDIDATE_ROOT.mkdir(parents=True, exist_ok=True)
_run_script(
    PROJECT_ROOT / "scripts/75_prepare_hst_acceptance.py",
    "--run-root", PILOT_RUN_ROOT,
    "--output", CANDIDATE_ROOT / "pilot_acceptance.candidate.json",
)
print("MANUAL_REVIEW_REQUIRED: review the pilot candidate. Promote only its proposed accepted document to reports/hst/accepted_freezes.json.")
print("Also review and commit the exact pilot environment audit as configs/hst_environment_lock.approved.json, then make it read-only.")


## Manual gate 1: pilot and environment

Review resource safety, AMP agreement, batch/accumulation, and the conservative full-run projection against the frozen 168-hour serial-GPU ceiling. The projection uses all contract-eligible Coswara participants, modality-specific job counts, and a 1.5 end-to-end overhead multiplier; it is a capacity estimate, not a completion-time guarantee. Also review the data-contract hash and Ubuntu dependency lock. Manually create `reports/hst/accepted_freezes.json` with approval identity and the three accepted hashes. Manually promote the exact reviewed environment audit to `configs/hst_environment_lock.approved.json`, commit it, and make it read-only. Do not continue if any pilot validity check failed.

In [ ]:
if not ACCEPTED_FREEZES_PATH.is_file() or not ENVIRONMENT_LOCK_PATH.is_file():
    raise FileNotFoundError("Manual gate 1 has not been promoted")
FULL_MANIFEST_LAUNCH, FULL_MANIFEST_STATUS = launch_and_wait(mode="full", through=THROUGH_MANIFESTS)
FULL_RUN_ID = FULL_MANIFEST_STATUS["run_id"]
FULL_RUN_ROOT = PROJECT_ROOT / "data/outputs/hst" / FULL_RUN_ID
FULL_MANIFEST_STATUS


In [ ]:
_run_script(
    PROJECT_ROOT / "scripts/76_prepare_hst_comparator_approval.py", "approval-record",
    "--project-root", PROJECT_ROOT.parent, "--run-root", FULL_RUN_ROOT,
    "--manifests-receipt", FULL_RUN_ROOT / "runtime/stages/manifests.json",
    "--feature-table", FEATURE_TABLE,
    "--pilot-accepted-freezes", ACCEPTED_FREEZES_PATH,
    "--environment-lock", ENVIRONMENT_LOCK_PATH, "--runtime-random-state", 42,
    "--output", CANDIDATE_ROOT / "comparator_approval.candidate.json",
)
print("MANUAL_REVIEW_REQUIRED: inspect and manually promote only the proposed approval record to the canonical approved path, commit it, and make it read-only.")


In [ ]:
if not APPROVAL_PATH.is_file():
    raise FileNotFoundError("Canonical comparator approval is not present")
_run_script(
    PROJECT_ROOT / "scripts/76_prepare_hst_comparator_approval.py", "accepted-freezes",
    "--project-root", PROJECT_ROOT.parent, "--approval-record", APPROVAL_PATH,
    "--pilot-accepted-freezes", ACCEPTED_FREEZES_PATH,
    "--environment-lock", ENVIRONMENT_LOCK_PATH, "--project-id", "covid-rars",
    "--expected-remote-url", EXPECTED_REMOTE_URL, "--runtime-random-state", 42,
    "--output", CANDIDATE_ROOT / "comparator_freezes.candidate.json",
)
print("MANUAL_REVIEW_REQUIRED: promote only proposed_accepted_freezes to the canonical comparator freezes path, commit it, and make it read-only.")


## Manual gate 2: comparator recipe

The approval must bind the exact aligned manifest, complete 10,147-column feature table, top-800 fold-local selection, four frozen model families, validation-selection rule, environment, source, and Git identity. The initial comparator accepted-freezes file has no accepted generation.

In [ ]:
if not COMPARATOR_FREEZES_PATH.is_file():
    raise FileNotFoundError("Manual gate 2 has not been promoted")
COMPARATOR_LAUNCH, COMPARATOR_STATUS = launch_and_wait(
    mode="full", through=THROUGH_COMPARATOR, expected_run_id=FULL_RUN_ID,
    expected_manual_gate="manual comparator generation acceptance required",
)
COMPARATOR_STATUS


In [ ]:
COMPARATOR_AUDIT = FULL_RUN_ROOT / "scientific/aligned_comparator/audit"
CURRENT_RECEIPT = COMPARATOR_AUDIT / "current.json"
CURRENT = json.loads(CURRENT_RECEIPT.read_text(encoding="ascii"))
GENERATION_MANIFEST = COMPARATOR_AUDIT / "generations" / CURRENT["generation_id"] / "manifest.json"
_run_script(
    PROJECT_ROOT / "scripts/77_prepare_hst_comparator_generation_acceptance.py",
    "--project-root", PROJECT_ROOT.parent, "--approval-record", APPROVAL_PATH,
    "--accepted-freezes", COMPARATOR_FREEZES_PATH,
    "--expected-accepted-freezes-sha256", _sha256(COMPARATOR_FREEZES_PATH),
    "--generation-manifest", GENERATION_MANIFEST, "--current-receipt", CURRENT_RECEIPT,
    "--runtime-random-state", 42,
    "--output", CANDIDATE_ROOT / "comparator_generation_acceptance.candidate.json",
)
print("MANUAL_REVIEW_REQUIRED: replace the canonical comparator freezes only with the proposed accepted-freezes update, commit it, and make it read-only.")


## Manual gate 3: generated comparator evidence

Verify the generation ID, manifest checksum, every model/table checksum, and approval bindings. Promote the reviewed update to the same canonical comparator accepted-freezes path. The final launch reuses this exact generation; it must not retrain or accept a different generation.

In [ ]:
FINAL_LAUNCH, FINAL_STATUS = launch_and_wait(
    mode="full", through=THROUGH_FINAL, expected_run_id=FULL_RUN_ID,
)
FINAL_STATUS


In [ ]:
LATEST_PATH = PROJECT_ROOT / "reports/hst/latest.json"
if not LATEST_PATH.is_file():
    raise FileNotFoundError("Verified evidence was not published")
LATEST = json.loads(LATEST_PATH.read_text(encoding="utf-8"))
SENSITIVITY_REGISTRY_PATH = FULL_RUN_ROOT / "contracts/sensitivity_execution_registry.csv"
if not SENSITIVITY_REGISTRY_PATH.is_file():
    raise FileNotFoundError("Sensitivity execution registry is missing")
SENSITIVITY_REGISTRY = pd.read_csv(SENSITIVITY_REGISTRY_PATH)
RAW_STATUS_METRICS_PATH = FULL_RUN_ROOT / "scientific/external_transfer/raw_status_sensitivity_metrics.csv"
RAW_STATUS_METRICS = pd.read_csv(RAW_STATUS_METRICS_PATH) if RAW_STATUS_METRICS_PATH.is_file() else None
{"latest": LATEST, "sensitivity_registry": SENSITIVITY_REGISTRY, "raw_status_metrics": RAW_STATUS_METRICS}
